# Licking by trial type, and what population anchoring tracks

Left and centre: where the animal licks on the virtual track, trial by trial, split by
**cued** (beaconed) and **uncued** (non-beaconed) trials. On cued trials the reward zone
is marked by a visual cue; on uncued trials its location can only be recovered by path
integration.

Right: the **population anchoring state** of the same session against performance on each
trial type. The prediction is that anchoring tracks *uncued* performance more closely
than cued — because a cued trial can be solved by looking, whether or not the internal
map is registered to the track, while an uncued trial cannot.

In [ ]:
import sys, os, glob, re, warnings
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pynapple as nap
from matplotlib.patches import Rectangle
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

from spatial_manifolds.anchoring import trial_cluster_labels, ANCH_COLOR, NONANCH_COLOR

SOURCE_PATH = '/Users/harryclark/Downloads/clark2025/'
FIGPATH = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
CLS = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications_v2.csv')

TL, BS = 200.0, 2.0                 # track length and position bin, cm
NBIN = int(TL / BS)
RZ = (90.0, 110.0)                  # reward zone: 30 cm black box + 60 cm track, 20 cm wide
BLACK_BOX = 30.0
CUED_C, UNCUED_C = '#c0398d', '#3f9b4f'     # matching the reference figure's two types
MIN_CELLS = 25

# Population anchoring is defined over MEC cells only. Without this filter it was
# computed over every spatial cell on the probe, which is 66% ENTm overall but
# ranges 0-100% per session -- 23 of 81 sessions were under half MEC, and 19 had
# no ENTm spatial cells at all. Those probes really did sit outside MEC (SUB,
# PRE, PAR, visual cortex, in a few cases cerebellum); anatomy is reconstructed
# for all of them, so this is genuine targeting, not a registration failure.
# Restricting costs 22 of 81 sessions, including all of M22, and leaves a median
# of 112 ENTm spatial cells per retained session.
REGION_PREFIX = 'ENTm'          # set to None to use every recorded spatial cell


def spatial_cell_ids(mouse, day, clusters, region=REGION_PREFIX,
                     classes=('GC', 'NGS')):
    """Cluster ids of the spatial cells that define the population state."""
    sel = CLS[(CLS.mouse == mouse) & (CLS.day == day) &
              (CLS.cell_class_of1.isin(classes))]
    if region is not None:
        sel = sel[sel.brain_region.astype(str).str.startswith(region)]
    return [int(c) for c in sel.cluster_id.astype(int) if int(c) in clusters.index]

def vr_paths(mouse, day):
    d = f'{SOURCE_PATH}M{mouse}/D{day}/VR/'
    stem = f'sub-M{mouse}_ses-D{day}_typ-VR'
    return d + stem + '_beh.nwb', d + stem + '_srt-kilosort4_clusters.npz'

def ephys_window(clusters):
    """Time span actually covered by the ephys recording.

    The behaviour clock starts before the ephys clock (t=0 is recording onset), so a
    session's trials table can contain dozens of trials that finished before the probe
    was recording. Those trials contain no spikes, every cell reads as silent, and the
    silent-trial guard forces them to 'non-anchored' -- a solid block at the top of the
    raster that is an alignment artefact, not a brain state. M29 D23 has 71 such
    trials; M28 D18 has 13 at the start and 24 after the recording ends.
    """
    last = [float(clusters[c].index[-1]) for c in clusters.index if len(clusters[c].index)]
    return 0.0, float(np.max(last))


def clip_trials(trials, clusters):
    """Keep only trials fully inside the ephys window, renumbered from 1."""
    t0, t1 = ephys_window(clusters)
    keep = (trials.start >= t0) & (trials.end <= t1)
    out = trials[keep].copy().reset_index(drop=True)
    out['orig_number'] = out.number.astype(int)
    out['number'] = np.arange(1, len(out) + 1)
    return out, out.orig_number.values


STOP_SPEED = 3.0          # cm/s -- the reward criterion used in the task

def load_vr(mouse, day):
    """Discrete STOP events: contiguous runs below the stop threshold, one point each.

    Every below-threshold sample would draw a dense smear wherever the animal
    lingers, which mostly reports dwell time rather than where it chose to stop.
    Taking one position per contiguous run counts stopping *events*, which is what
    the task rewards."""
    bp, cp = vr_paths(mouse, day)
    beh = nap.load_file(bp)
    clusters = nap.load_file(cp)
    trials, orig = clip_trials(beh['trials'].as_dataframe(), clusters)
    remap = {int(o): int(n) for o, n in zip(orig, trials.number)}
    P, tn, S = beh['P'], beh['trial_number'], beh['S']
    t = np.asarray(S.index); sp = np.asarray(S.values, float)
    pos_i = np.searchsorted(np.asarray(P.index), t).clip(0, len(P)-1)
    tr_i  = np.searchsorted(np.asarray(tn.index), t).clip(0, len(tn)-1)
    pos = np.asarray(P.values)[pos_i]; tr = np.asarray(tn.values)[tr_i]
    below = sp < STOP_SPEED
    edges = np.diff(below.astype(int))
    starts = np.where(edges == 1)[0] + 1
    ends   = np.where(edges == -1)[0] + 1
    if below[0]:  starts = np.r_[0, starts]
    if below[-1]: ends   = np.r_[ends, len(below)]
    n = min(len(starts), len(ends))
    rows = []
    for a, b in zip(starts[:n], ends[:n]):
        if b - a < 2:                      # ignore single-sample dips
            continue
        tr_orig = int(np.median(tr[a:b]))
        if tr_orig not in remap:        # stop belongs to a trial outside the recording
            continue
        rows.append(dict(t=t[a], pos=float(np.median(pos[a:b])),
                         trial=remap[tr_orig]))
    return beh, trials, pd.DataFrame(rows)

print('setup ready')

## Choosing an example session

The right-hand panel needs a session with (a) enough simultaneously recorded cells to
estimate a population state and (b) enough variation in performance to have something to
track. A session where the animal hits on every trial has no behavioural variance to
explain, so it cannot show anything either way.

In [ ]:
rows = []
for d in sorted(glob.glob(f'{SOURCE_PATH}M*/D*/VR')):
    m = re.search(r'M(\d+)/D(\d+)/VR', d)
    mouse, day = int(m.group(1)), int(m.group(2))
    bp, cp = vr_paths(mouse, day)
    if not (os.path.exists(bp) and os.path.exists(cp)):
        continue
    n_cells = len(CLS[(CLS.mouse == mouse) & (CLS.day == day) &
                      (CLS.cell_class_of1.isin(['GC', 'NGS']))])
    if n_cells < MIN_CELLS:
        continue
    try:
        t = nap.load_file(bp)['trials'].as_dataframe()
    except Exception:
        continue
    hit = (t.performance == 'hit')
    b, nb = t.type == 'b', t.type == 'nb'
    if b.sum() < 30 or nb.sum() < 30:
        continue
    rows.append(dict(mouse=mouse, day=day, n_cells=n_cells, n_trials=len(t),
                     hit_cued=hit[b].mean(), hit_uncued=hit[nb].mean(),
                     # variance in uncued outcome is what the right panel needs
                     var_uncued=hit[nb].std()))
INV = pd.DataFrame(rows).sort_values('var_uncued', ascending=False)
print(f'{len(INV)} VR sessions with >={MIN_CELLS} spatial cells and >=30 trials of each type\n')
print(INV.head(12).round(3).to_string(index=False))

## Licking by trial type

In [ ]:
def stop_raster(ax, stops, trials, ttype, color, title, t_first, t_last):
    """Stops for one trial type, plotted at their ACTUAL trial number so that both
    types share a single trial axis (indexing within type would give the two panels
    different, incomparable y ranges)."""
    keep = set(trials[trials.type == ttype].number.astype(int))
    L = stops[stops.trial.isin(keep)]
    ax.scatter(L.pos, L.trial, s=5, color=color, alpha=.8, linewidths=0)
    span = t_last - t_first
    ax.add_patch(Rectangle((RZ[0], t_first), RZ[1]-RZ[0], span,
                           color='#9ecae1', alpha=.45, zorder=0, lw=0))
    for x in (0, TL-BLACK_BOX):
        ax.add_patch(Rectangle((x, t_first), BLACK_BOX, span, color='0.9', zorder=0, lw=0))
    ax.set_xlim(0, TL); ax.set_ylim(t_last, t_first)
    ax.set_xlabel('Position (cm)'); ax.set_title(title, fontsize=10, color=color)
    ax.spines[['top','right']].set_visible(False)

def track_schematic(ax, cued):
    """Strip above each raster: black box, track, reward zone (cued = visibly marked)."""
    ax.add_patch(Rectangle((0,0), TL, 1, color='0.75', lw=0))
    for x in (0, TL-BLACK_BOX):
        ax.add_patch(Rectangle((x,0), BLACK_BOX, 1, color='0.35', lw=0))
    if cued:
        for x in np.arange(RZ[0], RZ[1], 2.6):
            ax.add_patch(Rectangle((x,0), 1.3, 1, color='k', lw=0))
    else:
        ax.add_patch(Rectangle((RZ[0],0), RZ[1]-RZ[0], 1, color='0.75', lw=0))
    ax.set_xlim(0, TL); ax.set_ylim(0,1); ax.axis('off')
print('plotting helpers ready')

## Population anchoring against performance on each trial type

In [ ]:
def population_anchoring(mouse, day, use_cache=True):
    """Fraction of spatial cells anchored on each trial.

    Prefers the precomputed labels in data/population_state/labels/, written by
    build_anchoring_labels.py. Every figure then rests on one set of labels
    rather than reclassifying independently, and the gated classifier's per-cell
    shuffle is paid once for the whole project instead of once per figure.
    Falls back to computing them if the cache is missing.
    """
    if use_cache:
        from spatial_manifolds.anchoring import load_session_labels
        z = load_session_labels(mouse, day)
        if z is not None:
            bp_, cp_ = vr_paths(mouse, day)
            tr = nap.load_file(bp_)['trials'].as_dataframe()
            tr = tr[tr.number.astype(int).isin(set(z['trial'].tolist()))]
            in_pop = np.isfinite(z['pc1_load'])
            return tr, z['labels'][in_pop], z['frac_anch']
    bp, cp = vr_paths(mouse, day)
    beh = nap.load_file(bp); clusters = nap.load_file(cp)
    trials_all = beh['trials'].as_dataframe()
    trials, orig = clip_trials(trials_all, clusters)
    keep_idx = np.isin(trials_all.number.values.astype(int), orig)
    tn = beh['trial_number']; trav = beh['travel']
    dt = trav - (float(np.asarray(tn.values)[0]) - 1) * TL
    moving = beh['S'].threshold(3.0, method='above').time_support
    n_tr = len(trials_all)          # tuning curves are built over ALL trials ...
    ids = spatial_cell_ids(mouse, day, clusters)   # MEC only -- see REGION_PREFIX
    tc = nap.compute_1d_tuning_curves(clusters[ids], dt, nb_bins=n_tr*NBIN,
                                      minmax=[0, n_tr*TL], ep=moving)
    labs = []
    for c in ids:
        M = np.asarray(tc[c]).reshape(n_tr, NBIN)[keep_idx]   # ... then clipped
        lab, _, _, _ = trial_cluster_labels(M)
        if not np.isnan(lab).all():
            labs.append(lab)
    L = np.array(labs)
    return trials, L, np.nanmean(L, axis=0)      # per-trial fraction anchored

def rolling(x, w=15):
    return pd.Series(x, dtype=float).rolling(w, center=True, min_periods=max(3, w//3)).mean().values
print('anchoring helper ready')

## The figure

In [ ]:
MOUSE, DAY = 25, 24
print(f'session: M{MOUSE} D{DAY}')
beh, trials, stops = load_vr(MOUSE, DAY)
trials_a, L, frac_anch = population_anchoring(MOUSE, DAY)
print(f'  {len(stops)} stop events over {len(trials)} trials')
print(f'  anchoring matrix: {L.shape[0]} cells x {L.shape[1]} trials; '
      f'mean fraction anchored {np.nanmean(frac_anch):.2f}')

# sort cells by their loading on PC1 of the cell x trial label matrix, so any
# shared structure appears as coherent horizontal bands rather than being
# scattered across an arbitrary cell ordering
Lc = L - np.nanmean(L, axis=1, keepdims=True)
Lc = np.nan_to_num(Lc)
U, sv, Vt = np.linalg.svd(Lc, full_matrices=False)
pc1_load = U[:, 0]
# Orient by correlation with the population anchored fraction. The loadings come
# from MEAN-CENTRED data, so their sign says which way a cell moves with the
# component and nothing about its baseline rate. Comparing the top-loading cells'
# anchored fraction against the population mean is a different quantity, and it
# flips the sign whenever those cells happen to sit below average (M26 D18: 0.486
# vs 0.543, giving corr -0.991 instead of +0.991 -- reversed cell ordering).
pc1_trial = Vt[0]
if np.corrcoef(pc1_trial, np.nan_to_num(frac_anch))[0, 1] < 0:
    pc1_load, pc1_trial = -pc1_load, -pc1_trial
order_cells = np.argsort(pc1_load)[::-1]
pc1_var = sv[0]**2 / np.sum(sv**2)
print(f'  PC1 explains {pc1_var:.3f} of label variance')

T_FIRST, T_LAST = 1, int(trials.number.max())
print(f'  clipped to the ephys window: {T_LAST} trials, renumbered from 1')
fig = plt.figure(figsize=(10, 4.6))
gs = fig.add_gridspec(2, 3, height_ratios=[.09, 1], width_ratios=[1, 1, 1.3],
                      hspace=.06, wspace=.22)
axes = []
for j, (tt, col, name, cued) in enumerate([('b', CUED_C, 'cued', True),
                                           ('nb', UNCUED_C, 'uncued', False)]):
    track_schematic(fig.add_subplot(gs[0, j]), cued)
    ax = fig.add_subplot(gs[1, j], sharey=axes[0] if axes else None)
    stop_raster(ax, stops, trials, tt, col, f'{name} trials', T_FIRST, T_LAST)
    if j == 0: ax.set_ylabel('Trial number')
    else: ax.tick_params(labelleft=False)
    axes.append(ax)

# transposed so trial number runs down the y axis like the rasters: the three
# panels then share one trial axis and events can be read across at the same height
from matplotlib.colors import ListedColormap, BoundaryNorm
TA_CMAP = ListedColormap([NONANCH_COLOR, ANCH_COLOR])
TA_NORM = BoundaryNorm([-0.5, 0.5, 1.5], TA_CMAP.N)
ax = fig.add_subplot(gs[1, 2], sharey=axes[0])
ax.imshow(L[order_cells].T, aspect='auto', cmap=TA_CMAP, norm=TA_NORM,
          interpolation='nearest', extent=[0, L.shape[0], T_LAST, T_FIRST])
ax.set_xlabel(f'cell (n={L.shape[0]}, sorted by PC1 loading)')
ax.tick_params(labelleft=False)
ax.set_title(f'population anchoring state\nPC1 explains {100*pc1_var:.0f}% of label variance',
             fontsize=9)
for sp_ in ax.spines.values(): sp_.set_visible(False)
axes.append(ax)
fig.suptitle(f'M{MOUSE} D{DAY}', fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGPATH}/stop_raster_anchoring.pdf', dpi=200, bbox_inches='tight')
plt.show()